# 🚀 Apigee X Provisioning - Global Control Plane (PSC or VPC Peering)

<a href="https://colab.research.google.com/github/gcp-samples/apigee-quickstart-notebook/blob/main/01_apigee_provisioning_global.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook automates end-to-end provisioning of an **Apigee X** organization on the **Global Control Plane**, with a choice of **two runtime networking modes** selected by `NETWORKING_MODE` in Step 0:

- **`PSC`** (default): Private Service Connect only, with `"disableVpcPeering": "true"`.
- **`VPC_PEERING`**: Service Networking VPC peering between your VPC and the Apigee tenant project, with `"disableVpcPeering": "false"` and `"authorizedNetwork": NETWORK_NAME`.

---

### 🌟 Why this Notebook? Choosing a Networking Mode
The Apigee provisioning wizard hides the networking trade-offs. This notebook makes the choice explicit. It then automates the whole setup for either mode: APIs, IP ranges, org, instance, northbound HTTPS load balancer, hostname and SSL.

1. **PSC mode (`disableVpcPeering: true`), recommended for new orgs**: No `/22` + `/28` CIDR allocations and no Service Networking peering. Northbound traffic flows Load Balancer → PSC Network Endpoint Group (NEG) → Apigee Service Attachment. Southbound, Apigee reaches private backends through PSC endpoint attachments. No VMs to run.
2. **VPC Peering mode (`disableVpcPeering: false`)**: The Apigee runtime is peered into your VPC, so API proxies can call private-IP backends in that VPC directly. You pay for this in two ways. First, a `/22` + `/28` of your address space is reserved. Second, a global external HTTPS LB cannot target a peered instance directly, so the notebook deploys a Managed Instance Group (MIG) of Envoy proxy VMs as the northbound bridge. You own those VMs and must patch and scale them.
3. **Same end state for both modes (automated SSL & wildcard hostname)**: Google-managed SSL certificates and free `nip.io` wildcard DNS are configured automatically, so your HTTPS endpoint is ready to test immediately.

#### ⚖️ Decision Table

| | PSC | VPC Peering |
|---|---|---|
| **IP ranges consumed** | None. No peering ranges are reserved; only a regional PSC NEG is created in your VPC. | A `/22` runtime range **plus** a separate `/28` support range, both reserved for Service Networking. |
| **Resizable later** | Nothing to resize. | **No.** An instance's IP ranges are fixed at creation. A different size means new ranges and a new instance, and a `/28` runtime range can never grow in place. |
| **Envoy/MIG bridge required** | **No.** LB → PSC NEG → Apigee service attachment. | **Yes.** The global external HTTPS LB targets an Envoy MIG in `$REGION`, which forwards to the instance's internal IP (`host`). |
| **Backend reachability model** | Apigee reaches backends through PSC **endpoint attachments** to published service attachments. Backends can be in any VPC or project, and overlapping IPs are OK. | Direct private-IP routing into the **peered VPC only**. Peering is **non-transitive**: VPCs peered to yours are unreachable, and on-prem backends need custom route export on the peering. |
| **Typical use case** | Greenfield orgs, backends spread across many VPCs or projects, and IP-constrained networks. | Brownfield estates with many private backends in one VPC, or orgs that must match an existing peered topology. |

---

### ⚠️ Critical Decisions Before You Begin

> [!WARNING]
> **Billing Type is Permanent & Cannot Be Changed Later!**
> - **`EVALUATION` (Trial)**: 60-day temporary trial org. **Cannot be upgraded or converted to a paid org later!** Not covered by Google Cloud SLAs and not supported for production.
> - **`PAYG` (Pay-As-You-Go)**: Standard production-ready pay-as-you-go billing based on API calls and analytics.
> - **`SUBSCRIPTION`**: For enterprise annual/multi-year subscription commitments.
> 
> *If you are planning for eventual production use, select `PAYG` or `SUBSCRIPTION`.*

> [!WARNING]
> **Networking Mode is Fixed at Org Creation & Cannot Be Changed Afterwards!**
> - `NETWORKING_MODE` sets `disableVpcPeering` on the Apigee organization, plus `authorizedNetwork` for VPC Peering. Once a runtime instance exists, the Apigee API rejects changes to either field. This notebook never tries to switch modes in place.
> - **To change modes you must delete and recreate the Apigee organization.**
> - If an org already exists and its mode contradicts your selection, Step 3 warns you, in either direction.
>
> *If you have no hard requirement for direct private-IP routing into a single VPC, select `PSC`.*

> [!WARNING]
> **A `/28` Peering Range is Minimal-Footprint & Cannot Be Expanded in Place!**
> - Reserved peering ranges and an instance's IP ranges cannot be resized after creation. Growing means allocating new ranges and creating a new instance.
> - An Apigee instance always needs one `/22` **and** one `/28`. With `PEERING_CIDR_SIZE = "/28"`, the `/22` must come from another range already attached to the Service Networking connection. Step 2 fails fast if no such range exists.
>
> *For anything heading to production, select `PEERING_CIDR_SIZE = "/22"`.*

---

### 🏗️ Architecture Overview

<table>
<tr>
<th>🔒 Mode A: <code>PSC</code> (disableVpcPeering = true)</th>
<th>🔗 Mode B: <code>VPC_PEERING</code> (disableVpcPeering = false)</th>
</tr>
<tr>
<td valign="top">
<pre>
Client (Internet)
       │
       ▼
[Global External HTTPS Load Balancer]
 (Port 443 + Google-managed SSL)
       │
       ▼
[PSC Network Endpoint Group (NEG)]
 (your VPC, $REGION)
       │ (Google Private Service Connect)
       ▼
[Apigee Service Attachment]
       │
       ▼
[Apigee Runtime Instance (Tenant Project)]
       │ southbound: PSC endpoint
       ▼ attachments
[Backends published via PSC
 service attachments (any VPC/project)]
</pre>
</td>
<td valign="top">
<pre>
Client (Internet)
       │
       ▼
[Global External HTTPS Load Balancer]
 (Port 443 + Google-managed SSL)
       │
       ▼
[Envoy MIG Bridge] (your VPC, $REGION)
 firewall: 130.211.0.0/22, 35.191.0.0/16
       │ HTTPS → instance host IP
       ▼ (Service Networking VPC peering)
[Apigee Runtime Instance (Tenant Project)]
 (/22 runtime + /28 support from your ranges)
       │ southbound: direct private-IP
       ▼ routing (non-transitive)
[Backends in your peered VPC]
</pre>
</td>
</tr>
</table>

## Step 0: Colab Authentication & Configuration

Run the cell below to authenticate your Google Cloud account and configure your deployment parameters.


In [ ]:
#@title ⚙️ Configuration Form [Run this cell]
#@markdown Fill in your configuration parameters below:

import os
import sys
import json
import time
import subprocess
import requests

# Authenticate in Google Colab
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Running in local/standard Jupyter environment.")

#@markdown #### 1. Google Cloud Project & Region
PROJECT_ID = "your-project-id" #@param {type:"string"}
REGION = "europe-west1" #@param ["europe-west1", "europe-west3", "europe-west4", "us-central1", "us-east1", "us-east4", "us-west1", "asia-east1", "asia-northeast1", "asia-southeast1", "australia-southeast1"]
ANALYTICS_REGION = "europe-west1" #@param ["europe-west1", "europe-west3", "europe-west4", "us-central1", "us-east1", "us-east4", "us-west1", "asia-east1", "asia-northeast1", "asia-southeast1", "australia-southeast1"]

#@markdown #### 2. Billing & Runtime Configuration
#@markdown *Important: EVALUATION cannot be upgraded to paid/production later.*
BILLING_TYPE = "EVALUATION" #@param ["EVALUATION", "PAYG", "SUBSCRIPTION"]
RUNTIME_TYPE = "CLOUD" #@param ["CLOUD"]

#@markdown #### 3. Network Selection & Networking Mode (VPC for PSC NEGs / VPC peering and project resources)
#@markdown *Important: NETWORKING_MODE is fixed at org creation. Switching later requires deleting and recreating the org.*
NETWORK_NAME = "default" #@param {type:"string"}
NETWORKING_MODE = "PSC" #@param ["PSC", "VPC_PEERING"]

#@markdown ##### 3a. VPC Peering Settings (used only when NETWORKING_MODE = "VPC_PEERING"; ignored for PSC)
#@markdown */22 is recommended. A /28 runtime range is minimal-footprint, cannot be expanded in place, and needs a /22 already available on the Service Networking connection.*
PEERING_CIDR_SIZE = "/22" #@param ["/22", "/28"]
PEERING_RANGE_NAME = "apigee-peering-range" #@param {type:"string"}
SUPPORT_RANGE_NAME = "apigee-support-range" #@param {type:"string"}

#@markdown #### 4. Apigee Environment & Instance Configuration
#@markdown *Leave INSTANCE_NAME empty or 'auto' to default to apigee-{REGION}*
INSTANCE_NAME = "auto" #@param {type:"string"}
ENV_NAME = "dev" #@param {type:"string"}
ENVGROUP_NAME = "dev-group" #@param {type:"string"}

# Compute default instance name if auto/empty
if not INSTANCE_NAME.strip() or INSTANCE_NAME.lower() == "auto":
    INSTANCE_NAME = f"apigee-{REGION}"

#@markdown #### 5. Optional Add-ons
ENABLE_MONETIZATION = True #@param {type:"boolean"}
ENABLE_API_SECURITY = True #@param {type:"boolean"}
ENABLE_ADVANCED_APIOPS = True #@param {type:"boolean"}

# Validate PROJECT_ID
if PROJECT_ID == "your-project-id" or not PROJECT_ID.strip():
    raise ValueError("❌ Please provide a valid Google Cloud PROJECT_ID in the form above!")

# Validate NETWORKING_MODE (guard: every mode-dependent cell also re-checks this)
SUPPORTED_NETWORKING_MODES = ("PSC", "VPC_PEERING")
NETWORKING_MODE = NETWORKING_MODE.strip()
if NETWORKING_MODE not in SUPPORTED_NETWORKING_MODES:
    raise ValueError(
        f"❌ Unrecognised NETWORKING_MODE '{NETWORKING_MODE}'. "
        f"Expected one of: {', '.join(SUPPORTED_NETWORKING_MODES)}."
    )

# Validate VPC peering parameters
if PEERING_CIDR_SIZE not in ("/22", "/28"):
    raise ValueError(f"❌ Unsupported PEERING_CIDR_SIZE '{PEERING_CIDR_SIZE}'. Expected '/22' or '/28'.")
PEERING_PREFIX_LENGTH = int(PEERING_CIDR_SIZE.lstrip("/"))
PEERING_RANGE_NAME = PEERING_RANGE_NAME.strip()
SUPPORT_RANGE_NAME = SUPPORT_RANGE_NAME.strip()

if NETWORKING_MODE == "VPC_PEERING":
    if not PEERING_RANGE_NAME or not SUPPORT_RANGE_NAME:
        raise ValueError("❌ PEERING_RANGE_NAME and SUPPORT_RANGE_NAME are required when NETWORKING_MODE = 'VPC_PEERING'.")
    if PEERING_RANGE_NAME == SUPPORT_RANGE_NAME:
        raise ValueError("❌ PEERING_RANGE_NAME and SUPPORT_RANGE_NAME must be different: Apigee needs a separate /28 support range.")
elif NETWORKING_MODE == "PSC":
    pass  # Peering parameters are ignored in PSC mode.
else:
    raise ValueError(f"❌ Unrecognised NETWORKING_MODE '{NETWORKING_MODE}'.")

# Set environment variables for shell commands
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION
os.environ["ANALYTICS_REGION"] = ANALYTICS_REGION
os.environ["BILLING_TYPE"] = BILLING_TYPE
os.environ["RUNTIME_TYPE"] = RUNTIME_TYPE
os.environ["NETWORK_NAME"] = NETWORK_NAME
os.environ["INSTANCE_NAME"] = INSTANCE_NAME
os.environ["ENV_NAME"] = ENV_NAME
os.environ["ENVGROUP_NAME"] = ENVGROUP_NAME
os.environ["NETWORKING_MODE"] = NETWORKING_MODE
os.environ["PEERING_CIDR_SIZE"] = PEERING_CIDR_SIZE
os.environ["PEERING_PREFIX_LENGTH"] = str(PEERING_PREFIX_LENGTH)
os.environ["PEERING_RANGE_NAME"] = PEERING_RANGE_NAME
os.environ["SUPPORT_RANGE_NAME"] = SUPPORT_RANGE_NAME

# Set gcloud default project
!gcloud config set project $PROJECT_ID --quiet

print(f"\n🎯 Configured for Project: {PROJECT_ID}")
print(f"🌍 Instance: {INSTANCE_NAME} in Region: {REGION} | Analytics: {ANALYTICS_REGION}")
print(f"💳 Billing Type: {BILLING_TYPE} | Runtime Type: {RUNTIME_TYPE}")
print(f"🌐 VPC Network: {NETWORK_NAME}")
print(f"🔀 Networking Mode: {NETWORKING_MODE}")
if NETWORKING_MODE == "VPC_PEERING":
    print(f"📐 Peering Ranges: {PEERING_RANGE_NAME} ({PEERING_CIDR_SIZE}) + {SUPPORT_RANGE_NAME} (/28 support)")
    if PEERING_CIDR_SIZE == "/28":
        print("   ⚠️ /28 runtime range selected: minimal footprint, cannot be expanded in place. Use /22 for production.")
elif NETWORKING_MODE == "PSC":
    print(f"📐 Peering Ranges: none consumed (PEERING_CIDR_SIZE={PEERING_CIDR_SIZE}, "
          f"{PEERING_RANGE_NAME}, {SUPPORT_RANGE_NAME} are ignored in PSC mode)")
else:
    raise ValueError(f"❌ Unrecognised NETWORKING_MODE '{NETWORKING_MODE}'.")
print(f"✨ Addons: Monetization={ENABLE_MONETIZATION}, ApiSecurity={ENABLE_API_SECURITY}, AdvancedApiOps={ENABLE_ADVANCED_APIOPS}")

In [ ]:
# Helper function for Google API requests and long-running operation polling
def get_auth_token():
    token = subprocess.check_output(
        ["gcloud", "auth", "print-access-token"], text=True
    ).strip()
    return token

def apigee_headers():
    return {
        "Authorization": f"Bearer {get_auth_token()}",
        "Content-Type": "application/json; charset=utf-8"
    }

def poll_apigee_operation(op_url, description="Operation", poll_interval=15, max_timeout=3600):
    start_time = time.time()
    print(f"⏳ Waiting for {description} to complete...")
    while time.time() - start_time < max_timeout:
        resp = requests.get(op_url, headers=apigee_headers())
        if resp.status_code == 200:
            data = resp.json()
            if data.get("done", False):
                if "error" in data:
                    raise RuntimeError(f"❌ {description} failed: {json.dumps(data['error'], indent=2)}")
                elapsed = int(time.time() - start_time)
                print(f"✅ {description} completed successfully in {elapsed // 60}m {elapsed % 60}s!")
                return data.get("response", {})
        elapsed = int(time.time() - start_time)
        mins, secs = divmod(elapsed, 60)
        print(f"   ⏱️ Still in progress... ({mins}m {secs:02d}s elapsed)", end="\r")
        time.sleep(poll_interval)
    raise TimeoutError(f"❌ Timed out waiting for {description} after {max_timeout // 60} minutes.")

def run_gcloud(cmd, check=True):
    """Runs a shell (gcloud) command and returns the CompletedProcess.

    Raises RuntimeError with stderr when check=True and the command fails.
    """
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"❌ Command failed (exit {result.returncode}): {cmd}\n{result.stderr.strip()}")
    return result

def unsupported_mode_error():
    """Guard used by every NETWORKING_MODE branch for unrecognised values."""
    return ValueError(
        f"❌ Unrecognised NETWORKING_MODE '{NETWORKING_MODE}'. "
        f"Expected one of: {', '.join(SUPPORTED_NETWORKING_MODES)}. Re-run the Configuration Form."
    )

print("🛠️ Apigee helper utilities initialized.")

## Step 1: Enable Google Cloud APIs

We enable the required APIs:
- **Apigee API** (`apigee.googleapis.com`)
- **Compute Engine API** (`compute.googleapis.com`)
- **API Hub** (`apihub.googleapis.com`)
- **Cloud KMS** (`cloudkms.googleapis.com`)
- **Service Networking API** (`servicenetworking.googleapis.com`): **`VPC_PEERING` mode only.** It brokers the VPC peering between your VPC and the Apigee tenant project. PSC mode does not use it.

In [ ]:
#@title 🔌 Enable APIs
REQUIRED_APIS = [
    "apigee.googleapis.com",
    "compute.googleapis.com",
    "apihub.googleapis.com",
    "cloudkms.googleapis.com",
]

if NETWORKING_MODE == "VPC_PEERING":
    # Service Networking brokers the VPC peering with the Apigee tenant project.
    REQUIRED_APIS.append("servicenetworking.googleapis.com")
elif NETWORKING_MODE == "PSC":
    pass  # PSC does not use Service Networking.
else:
    raise unsupported_mode_error()

APIS_TO_ENABLE = " ".join(REQUIRED_APIS)
!gcloud services enable $APIS_TO_ENABLE --project $PROJECT_ID

print(f"✅ Required Google Cloud APIs enabled for NETWORKING_MODE={NETWORKING_MODE}:")
for api in REQUIRED_APIS:
    print(f"   • {api}")

## Step 2: Network & IP Range Allocation

**Both modes:** Check that the selected VPC network (e.g., `default`) exists. If your project doesn't have it, this step creates it (auto subnet mode).

**`VPC_PEERING` only.** These extra sub-steps reserve address space for Apigee and peer it into your VPC:
- **(a) Runtime range:** allocate `$PEERING_RANGE_NAME` (`/22` or `/28`, from `PEERING_CIDR_SIZE`) with `--purpose=VPC_PEERING`.
- **(b) Support range:** allocate a **separate `/28`** range, `$SUPPORT_RANGE_NAME`. Apigee requires it *in addition to* the runtime range.
- **(d) Pre-flight validation:** check that the allocated blocks don't overlap existing subnets (primary & secondary), other reserved ranges, or routes on the network. This runs *before* the peering is created, and the cell fails fast with a clear message on any overlap. Ranges this run just created are released on failure. Existing ranges with the wrong size, network or purpose are rejected, because ranges cannot be resized in place.
- **(c) Peering:** `gcloud services vpc-peerings connect --service=servicenetworking.googleapis.com --network=$NETWORK_NAME --ranges=$PEERING_RANGE_NAME,$SUPPORT_RANGE_NAME`. If the network already has a Service Networking connection (e.g., from Cloud SQL), the ranges are **appended** with `vpc-peerings update`, so the existing ranges stay in place.
- **(e) Output:** print the allocated CIDR blocks before continuing.
- **Bridge prerequisite:** confirm the network has a subnet in `$REGION` and enable **Private Google Access** on it. The Envoy MIG bridge in Step 5 has no external IPs.

**`PSC`:** all peering sub-steps are skipped, and no peering IP ranges are consumed.

In [ ]:
#@title 🌐 Verify / Create VPC Network & Allocate IP Ranges
import ipaddress

# ---------------------------------------------------------------------------
# 2.1 Verify / create the VPC network (both modes)
# ---------------------------------------------------------------------------
net_check = subprocess.run(
    f"gcloud compute networks describe {NETWORK_NAME} --project={PROJECT_ID} --format='value(name)'",
    shell=True, capture_output=True, text=True
)

if net_check.returncode != 0:
    print(f"VPC network '{NETWORK_NAME}' does not exist in project '{PROJECT_ID}'. Creating it now...")
    !gcloud compute networks create $NETWORK_NAME --project=$PROJECT_ID --subnet-mode=auto
    print(f"✅ Created VPC network '{NETWORK_NAME}'.")
else:
    print(f"✅ VPC network '{NETWORK_NAME}' exists and is ready to use.")

PEERING_CIDR = None
SUPPORT_CIDR = None
MIG_SUBNET = None

if NETWORKING_MODE == "PSC":
    print("\n🔒 NETWORKING_MODE=PSC: skipping IP range allocation and Service Networking VPC peering.")
    print("   No peering IP ranges are consumed from your VPC (disableVpcPeering=true).")

elif NETWORKING_MODE == "VPC_PEERING":
    SERVICE_NETWORKING_PEERING = "servicenetworking-googleapis-com"

    def _network_matches(resource_url):
        return (resource_url or "").rstrip("/").split("/")[-1] == NETWORK_NAME

    def _cidr_of(addr):
        return ipaddress.ip_network(f"{addr['address']}/{addr['prefixLength']}", strict=False)

    def _describe_range(name):
        res = run_gcloud(
            f"gcloud compute addresses describe {name} --global --project={PROJECT_ID} --format=json",
            check=False,
        )
        return json.loads(res.stdout) if res.returncode == 0 and res.stdout.strip() else None

    def _list_json(cmd):
        out = run_gcloud(cmd).stdout.strip()
        return json.loads(out) if out else []

    # -----------------------------------------------------------------------
    # 2.2 Inspect the existing Service Networking connection (if any)
    # -----------------------------------------------------------------------
    print(f"\n2.2 Inspecting Service Networking connection on network '{NETWORK_NAME}'...")
    conn_res = run_gcloud(
        f"gcloud services vpc-peerings list --network={NETWORK_NAME} "
        f"--service=servicenetworking.googleapis.com --project={PROJECT_ID} --format=json",
        check=False,
    )
    connections = json.loads(conn_res.stdout) if conn_res.returncode == 0 and conn_res.stdout.strip() else []
    existing_conn_ranges = []
    for conn in connections:
        existing_conn_ranges.extend(conn.get("reservedPeeringRanges", []) or [])
    if connections:
        print(f"   Existing connection found with ranges: {', '.join(existing_conn_ranges) or '(none)'}")
    else:
        print("   No existing Service Networking connection; one will be created.")

    all_global_addresses = _list_json(f"gcloud compute addresses list --global --project={PROJECT_ID} --format=json")
    addresses_by_name = {a.get("name"): a for a in all_global_addresses}

    # /28 runtime range feasibility: the instance still needs a /22 from somewhere.
    if PEERING_PREFIX_LENGTH == 28:
        larger_ranges = [
            name for name in existing_conn_ranges
            if name not in (PEERING_RANGE_NAME, SUPPORT_RANGE_NAME)
            and int(addresses_by_name.get(name, {}).get("prefixLength") or 99) <= 22
        ]
        if not larger_ranges:
            raise ValueError(
                "❌ PEERING_CIDR_SIZE='/28' cannot be satisfied: an Apigee instance always needs one /22 AND one /28,\n"
                f"   and the Service Networking connection on '{NETWORK_NAME}' has no other /22-or-larger range to supply the /22.\n"
                "   → Re-run the Configuration Form with PEERING_CIDR_SIZE='/22' (recommended), or first attach a /22 (or larger)\n"
                "     VPC_PEERING range to the connection."
            )
        print(f"   ℹ️ /28 runtime range: Apigee will take its /22 from existing range(s): {', '.join(larger_ranges)}")

    # -----------------------------------------------------------------------
    # 2.3 (a) + (b) Allocate the runtime range and the separate /28 support range
    # -----------------------------------------------------------------------
    newly_created = []

    def _ensure_range(name, prefix_length, description):
        existing = _describe_range(name)
        if existing:
            if existing.get("purpose") != "VPC_PEERING":
                raise ValueError(
                    f"❌ Global address '{name}' already exists with purpose={existing.get('purpose')} (expected VPC_PEERING). "
                    "Choose a different range name in the Configuration Form."
                )
            if not _network_matches(existing.get("network")):
                raise ValueError(
                    f"❌ Range '{name}' already exists on network '{(existing.get('network') or '').split('/')[-1]}', "
                    f"not '{NETWORK_NAME}'. Choose a different range name."
                )
            if int(existing.get("prefixLength") or 0) != prefix_length:
                raise ValueError(
                    f"❌ Range '{name}' already exists as /{existing.get('prefixLength')} but /{prefix_length} was requested.\n"
                    "   Allocated ranges cannot be resized in place. Delete it if unused "
                    f"(gcloud compute addresses delete {name} --global --project={PROJECT_ID}) or choose a different name."
                )
            print(f"   ℹ️ Range '{name}' already exists: {_cidr_of(existing)} (reusing).")
            return existing
        print(f"   Allocating /{prefix_length} range '{name}' on network '{NETWORK_NAME}'...")
        run_gcloud(
            f"gcloud compute addresses create {name} --global --purpose=VPC_PEERING "
            f"--prefix-length={prefix_length} --network={NETWORK_NAME} "
            f"--description='{description}' --project={PROJECT_ID} --quiet"
        )
        newly_created.append(name)
        created = _describe_range(name)
        print(f"   ✅ Allocated '{name}': {_cidr_of(created)}")
        return created

    print(f"\n2.3a Runtime peering range '{PEERING_RANGE_NAME}' ({PEERING_CIDR_SIZE})...")
    peering_addr = _ensure_range(PEERING_RANGE_NAME, PEERING_PREFIX_LENGTH, "Peering range for Apigee runtime")
    print(f"\n2.3b Support range '{SUPPORT_RANGE_NAME}' (/28)...")
    support_addr = _ensure_range(SUPPORT_RANGE_NAME, 28, "Peering range for supporting Apigee services")

    # -----------------------------------------------------------------------
    # 2.4 (d) Pre-flight: no overlap with subnets, reserved ranges or routes
    # -----------------------------------------------------------------------
    print("\n2.4 Pre-flight: checking for overlaps with existing subnets, reserved ranges and routes...")
    our_ranges = {PEERING_RANGE_NAME: _cidr_of(peering_addr), SUPPORT_RANGE_NAME: _cidr_of(support_addr)}
    existing_blocks = []

    subnets = _list_json(f"gcloud compute networks subnets list --network={NETWORK_NAME} --project={PROJECT_ID} --format=json")
    for s in subnets:
        s_region = (s.get("region") or "").split("/")[-1]
        existing_blocks.append((f"subnet '{s['name']}' ({s_region}) primary", s.get("ipCidrRange")))
        for sec in s.get("secondaryIpRanges", []) or []:
            existing_blocks.append((f"subnet '{s['name']}' ({s_region}) secondary '{sec.get('rangeName')}'", sec.get("ipCidrRange")))

    for a in all_global_addresses:
        if a.get("name") in our_ranges or not _network_matches(a.get("network")):
            continue
        prefix = a.get("prefixLength") or 32
        existing_blocks.append((f"reserved range '{a['name']}' ({a.get('purpose')})", f"{a.get('address')}/{prefix}"))

    routes = _list_json(f"gcloud compute routes list --project={PROJECT_ID} --format=json")
    for r in routes:
        if not _network_matches(r.get("network")):
            continue
        dest = r.get("destRange")
        if not dest or dest == "0.0.0.0/0" or r.get("nextHopNetwork"):
            continue  # default route / subnet routes (already covered by subnets)
        if (r.get("nextHopPeering") or "") == SERVICE_NETWORKING_PEERING:
            continue  # routes exported back by our own Service Networking peering
        existing_blocks.append((f"route '{r['name']}'", dest))

    conflicts = []
    if our_ranges[PEERING_RANGE_NAME].overlaps(our_ranges[SUPPORT_RANGE_NAME]):
        conflicts.append(f"   • {PEERING_RANGE_NAME} ({our_ranges[PEERING_RANGE_NAME]}) overlaps {SUPPORT_RANGE_NAME} ({our_ranges[SUPPORT_RANGE_NAME]})")
    for label, cidr in existing_blocks:
        try:
            other = ipaddress.ip_network(cidr, strict=False)
        except (TypeError, ValueError):
            continue
        for name, ours in our_ranges.items():
            if ours.version == other.version and ours.overlaps(other):
                conflicts.append(f"   • {name} ({ours}) overlaps {label} ({other})")

    if conflicts:
        for name in newly_created:
            print(f"   🧹 Releasing range '{name}' created by this run...")
            run_gcloud(f"gcloud compute addresses delete {name} --global --project={PROJECT_ID} --quiet", check=False)
        raise ValueError(
            "❌ Pre-flight failed: allocated peering ranges overlap existing address space on "
            f"network '{NETWORK_NAME}':\n" + "\n".join(conflicts) + "\n"
            "   → Pre-create the range(s) on a free block with the same name(s), e.g.\n"
            f"     gcloud compute addresses create {PEERING_RANGE_NAME} --global --purpose=VPC_PEERING "
            f"--addresses=<FREE_BLOCK> --prefix-length={PEERING_PREFIX_LENGTH} --network={NETWORK_NAME}\n"
            "     and re-run this cell. Existing ranges with matching size are reused."
        )
    print(f"   ✅ No overlaps across {len(existing_blocks)} existing subnet/range/route block(s).")

    # -----------------------------------------------------------------------
    # 2.5 (c) Create or extend the Service Networking VPC peering
    # -----------------------------------------------------------------------
    print(f"\n2.5 Establishing Service Networking VPC peering on '{NETWORK_NAME}'...")
    if not connections:
        run_gcloud(
            f"gcloud services vpc-peerings connect --service=servicenetworking.googleapis.com "
            f"--network={NETWORK_NAME} --ranges={PEERING_RANGE_NAME},{SUPPORT_RANGE_NAME} "
            f"--project={PROJECT_ID}"
        )
        print("   ✅ VPC peering connection created.")
    else:
        missing = [r for r in (PEERING_RANGE_NAME, SUPPORT_RANGE_NAME) if r not in existing_conn_ranges]
        if missing:
            merged = existing_conn_ranges + missing  # keep existing ranges: never drop ranges in use
            run_gcloud(
                f"gcloud services vpc-peerings update --service=servicenetworking.googleapis.com "
                f"--network={NETWORK_NAME} --ranges={','.join(merged)} --project={PROJECT_ID}"
            )
            print(f"   ✅ Existing connection updated; added range(s): {', '.join(missing)}")
        else:
            print("   ℹ️ Both ranges are already attached to the Service Networking connection.")

    # -----------------------------------------------------------------------
    # 2.6 (e) Print the resulting allocated CIDR blocks
    # -----------------------------------------------------------------------
    PEERING_CIDR = str(our_ranges[PEERING_RANGE_NAME])
    SUPPORT_CIDR = str(our_ranges[SUPPORT_RANGE_NAME])
    os.environ["PEERING_CIDR"] = PEERING_CIDR
    os.environ["SUPPORT_CIDR"] = SUPPORT_CIDR

    final_conn = run_gcloud(
        f"gcloud services vpc-peerings list --network={NETWORK_NAME} "
        f"--service=servicenetworking.googleapis.com --project={PROJECT_ID} --format=json",
        check=False,
    )
    final_ranges = []
    if final_conn.returncode == 0 and final_conn.stdout.strip():
        for conn in json.loads(final_conn.stdout):
            final_ranges.extend(conn.get("reservedPeeringRanges", []) or [])

    print("\n2.6 Allocated CIDR blocks:")
    print(f"   📐 Runtime range  {PEERING_RANGE_NAME:<28} {PEERING_CIDR}")
    print(f"   🧰 Support range  {SUPPORT_RANGE_NAME:<28} {SUPPORT_CIDR}")
    print(f"   🔗 Ranges on the Service Networking connection: {', '.join(final_ranges) or '(unknown)'}")

    # -----------------------------------------------------------------------
    # 2.7 Bridge prerequisite: subnet in $REGION with Private Google Access
    # -----------------------------------------------------------------------
    print(f"\n2.7 Checking for a subnet in {REGION} for the Envoy MIG bridge (Step 5)...")
    region_subnets = [s for s in subnets if (s.get("region") or "").split("/")[-1] == REGION]
    if not region_subnets:
        raise ValueError(
            f"❌ Network '{NETWORK_NAME}' has no subnet in {REGION}. The VPC-peering northbound bridge needs one.\n"
            f"   → gcloud compute networks subnets create {NETWORK_NAME}-{REGION} --network={NETWORK_NAME} "
            f"--region={REGION} --range=<FREE_CIDR> --enable-private-ip-google-access --project={PROJECT_ID}"
        )
    MIG_SUBNET = region_subnets[0]["name"]
    os.environ["MIG_SUBNET"] = MIG_SUBNET
    if not region_subnets[0].get("privateIpGoogleAccess"):
        run_gcloud(
            f"gcloud compute networks subnets update {MIG_SUBNET} --region={REGION} "
            f"--enable-private-ip-google-access --project={PROJECT_ID}"
        )
        print(f"   ✅ Enabled Private Google Access on subnet '{MIG_SUBNET}'.")
    else:
        print(f"   ✅ Subnet '{MIG_SUBNET}' already has Private Google Access.")

    print("\n✅ VPC peering ranges allocated and Service Networking peering is active.")

else:
    raise unsupported_mode_error()

## Step 3: Create Apigee Organization (Global Control Plane)

We will now create the Apigee Organization. The networking fields depend on `NETWORKING_MODE`:

| `NETWORKING_MODE` | `disableVpcPeering` | `authorizedNetwork` |
|---|---|---|
| `PSC` | `"true"` | *(omitted)* |
| `VPC_PEERING` | `"false"` | `NETWORK_NAME` |

> [!NOTE]
> The Organization API has **no field for peering range names**. Apigee uses the ranges attached to the Service Networking connection on `authorizedNetwork`, which Step 2 set up. Step 4 then pins the exact CIDRs through the instance's `ipRange` field.

If the org already exists, the cell compares its `disableVpcPeering` value with the selected `NETWORKING_MODE` and **warns on a mismatch in either direction**. The mode cannot be changed after creation. To switch, delete and recreate the org.

> [!NOTE]
> Creating an Apigee organization takes approximately **5 to 10 minutes**. The script will automatically poll until the organization state is `ACTIVE`.

In [ ]:
#@title 🏛️ Create Apigee Organization
# Check if org already exists
org_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}"
org_check = requests.get(org_url, headers=apigee_headers())

# Networking settings implied by the selected mode
if NETWORKING_MODE == "PSC":
    desired_peering_disabled = True
elif NETWORKING_MODE == "VPC_PEERING":
    desired_peering_disabled = False
else:
    raise unsupported_mode_error()

if org_check.status_code == 200 and org_check.json().get("state") == "ACTIVE":
    existing_org = org_check.json()
    peering_disabled = existing_org.get("disableVpcPeering", False)
    if isinstance(peering_disabled, str):
        peering_disabled = peering_disabled.lower() == "true"
    existing_network = existing_org.get("authorizedNetwork", "") or ""
    existing_mode = "PSC" if peering_disabled else "VPC_PEERING"
    print(f"✅ Apigee Organization '{PROJECT_ID}' already exists and is ACTIVE!")
    print(f"   disableVpcPeering: {peering_disabled}  (→ {existing_mode} mode)")
    print(f"   authorizedNetwork: {existing_network or '(none)'}")

    if peering_disabled != desired_peering_disabled:
        print("\n" + "⚠️ " * 20)
        print(f"⚠️ WARNING: NETWORKING MODE MISMATCH! You selected NETWORKING_MODE='{NETWORKING_MODE}',")
        print(f"   but this existing org was created in '{existing_mode}' mode (disableVpcPeering={peering_disabled}).")
        print("   The networking mode is fixed at org creation: it CANNOT be changed after creation")
        print("   (the Apigee API rejects the change once runtime instances exist, and this notebook never attempts it).")
        print("   To use the selected mode, the org must be DELETED and RECREATED.")
        print(f"   Otherwise, re-run the Configuration Form with NETWORKING_MODE='{existing_mode}' to match this org.")
        print("⚠️ " * 20)
    elif NETWORKING_MODE == "VPC_PEERING" and existing_network.split("/")[-1] != NETWORK_NAME:
        print(f"\n⚠️ WARNING: This org is peered with network '{existing_network}', but NETWORK_NAME='{NETWORK_NAME}'.")
        print("   authorizedNetwork is fixed once runtime instances exist. Re-run the Configuration Form with the")
        print("   org's network, or delete and recreate the org.")
else:
    org_payload = {
        "displayName": PROJECT_ID,
        "description": f"Apigee Org for {PROJECT_ID}",
        "analyticsRegion": ANALYTICS_REGION,
        "runtimeType": RUNTIME_TYPE,
        "billingType": BILLING_TYPE,
        "addonsConfig": {
            "monetizationConfig": {"enabled": ENABLE_MONETIZATION},
            "advancedApiOpsConfig": {"enabled": ENABLE_ADVANCED_APIOPS},
            "apiSecurityConfig": {"enabled": ENABLE_API_SECURITY}
        },
        "portalDisabled": True
    }

    if NETWORKING_MODE == "PSC":
        # Current behaviour: pure PSC, no authorizedNetwork.
        org_payload["disableVpcPeering"] = "true"
        print(f"Creating Apigee Organization '{PROJECT_ID}' with billingType={BILLING_TYPE} and disableVpcPeering=true...")
    elif NETWORKING_MODE == "VPC_PEERING":
        # Fail fast if Step 2 has not established the Service Networking peering.
        peer_check = run_gcloud(
            f"gcloud services vpc-peerings list --network={NETWORK_NAME} "
            f"--service=servicenetworking.googleapis.com --project={PROJECT_ID} --format=json",
            check=False,
        )
        if peer_check.returncode != 0 or not json.loads(peer_check.stdout or "[]"):
            raise RuntimeError(
                f"❌ No Service Networking peering found on network '{NETWORK_NAME}'. Run Step 2 first."
            )
        org_payload["disableVpcPeering"] = "false"
        org_payload["authorizedNetwork"] = NETWORK_NAME
        # NOTE: The Organization resource has no field for peering range names. Apigee uses the ranges
        # attached to the Service Networking connection on authorizedNetwork (Step 2):
        #   PEERING_RANGE_NAME / SUPPORT_RANGE_NAME. Exact CIDRs are pinned per instance via `ipRange` (Step 4).
        print(f"Creating Apigee Organization '{PROJECT_ID}' with billingType={BILLING_TYPE}, "
              f"disableVpcPeering=false and authorizedNetwork={NETWORK_NAME}...")
        print(f"   Service Networking ranges: {PEERING_RANGE_NAME} ({PEERING_CIDR_SIZE}), {SUPPORT_RANGE_NAME} (/28)")
    else:
        raise unsupported_mode_error()

    create_resp = requests.post(
        f"https://apigee.googleapis.com/v1/organizations?parent=projects/{PROJECT_ID}",
        headers=apigee_headers(),
        json=org_payload
    )
    
    if create_resp.status_code not in (200, 201):
        print(f"Response: {create_resp.status_code} - {create_resp.text}")
        if "already exists" in create_resp.text:
            print("Org creation is already in progress. Waiting for ACTIVE state...")
        else:
            create_resp.raise_for_status()
    else:
        op_data = create_resp.json()
        op_name = op_data.get("name")
        if op_name:
            poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op_name}", "Organization Creation")

    # Double check state
    start_time = time.time()
    while time.time() - start_time < 900:
        resp = requests.get(org_url, headers=apigee_headers())
        if resp.status_code == 200 and resp.json().get("state") == "ACTIVE":
            print(f"✅ Apigee Organization '{PROJECT_ID}' is ACTIVE!")
            break
        print("   ⏱️ Waiting for organization to reach ACTIVE state...", end="\r")
        time.sleep(15)

## Step 4: Create Apigee Runtime Instance

Now we provision the Apigee runtime instance in your selected region. This step runs in both modes; only the output we extract differs:

- **`PSC`**: waits for the instance's **`serviceAttachment`** URI, which Step 5 uses for the PSC NEG. The cell fails if the URI is missing.
- **`VPC_PEERING`**: when `PEERING_CIDR_SIZE = "/22"`, pins the instance's `ipRange` to the Step 2 CIDRs (`<runtime>/22,<support>/28`). It then waits for the instance **`host`**, the internal runtime IP inside your peered range. That IP is the upstream for the Envoy MIG bridge in Step 5. A missing `serviceAttachment` is fine in this mode.

> [!IMPORTANT]
> Instance creation provisions Google-managed compute resources in the tenant project and typically takes **20 to 30 minutes**.
> An instance's IP ranges cannot be changed after creation.

In [ ]:
#@title 💻 Provision Apigee Runtime Instance
instance_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances/{INSTANCE_NAME}"
inst_resp = requests.get(instance_url, headers=apigee_headers())

TARGET_SERVICE = None
INSTANCE_HOST = None

if NETWORKING_MODE == "PSC":
    REQUIRED_INSTANCE_FIELD = "serviceAttachment"
elif NETWORKING_MODE == "VPC_PEERING":
    REQUIRED_INSTANCE_FIELD = "host"
else:
    raise unsupported_mode_error()

def extract_instance_endpoints(inst_data):
    """Stores serviceAttachment/host; returns True when the mode's required field is present."""
    global TARGET_SERVICE, INSTANCE_HOST
    TARGET_SERVICE = inst_data.get("serviceAttachment")
    INSTANCE_HOST = inst_data.get("host")
    return bool(inst_data.get(REQUIRED_INSTANCE_FIELD))

if inst_resp.status_code == 200 and inst_resp.json().get("state") == "ACTIVE":
    inst_data = inst_resp.json()
    extract_instance_endpoints(inst_data)
    print(f"✅ Instance '{INSTANCE_NAME}' already exists and is ACTIVE.")
else:
    print(f"Creating Apigee Instance '{INSTANCE_NAME}' in region '{REGION}'...")
    inst_payload = {
        "name": INSTANCE_NAME,
        "location": REGION,
        "description": f"Apigee instance in {REGION}",
        "displayName": f"Instance {REGION}"
    }

    if NETWORKING_MODE == "PSC":
        pass  # No IP ranges: PSC orgs do not consume peering ranges.
    elif NETWORKING_MODE == "VPC_PEERING":
        if PEERING_PREFIX_LENGTH == 22:
            if not (globals().get("PEERING_CIDR") and globals().get("SUPPORT_CIDR")):
                raise RuntimeError("❌ PEERING_CIDR / SUPPORT_CIDR are not set. Run Step 2 first.")
            # Pin the exact /22 + /28 reserved in Step 2 (ranges are immutable once the instance exists).
            inst_payload["ipRange"] = f"{PEERING_CIDR},{SUPPORT_CIDR}"
            print(f"   ipRange pinned to {inst_payload['ipRange']} ({PEERING_RANGE_NAME} + {SUPPORT_RANGE_NAME})")
        else:
            # /28 runtime range: the /22 must come from another connection range (validated in Step 2).
            print("   /28 runtime range selected: letting Apigee pick its /22 + /28 from the Service Networking connection.")
    else:
        raise unsupported_mode_error()
    
    create_inst = requests.post(
        f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances",
        headers=apigee_headers(),
        json=inst_payload
    )
    
    if create_inst.status_code not in (200, 201):
        print(f"Instance create response: {create_inst.status_code} - {create_inst.text}")
        if "already exists" not in create_inst.text:
            create_inst.raise_for_status()
    else:
        op_name = create_inst.json().get("name")
        if op_name:
            poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op_name}", "Instance Creation", poll_interval=20, max_timeout=3600)
    
    # Poll instance details until ACTIVE and the mode-specific field is available
    print(f"Fetching instance {REQUIRED_INSTANCE_FIELD}...")
    start_time = time.time()
    while time.time() - start_time < 3600:
        inst_resp = requests.get(instance_url, headers=apigee_headers())
        if inst_resp.status_code == 200:
            inst_data = inst_resp.json()
            if inst_data.get("state") == "ACTIVE" and extract_instance_endpoints(inst_data):
                print(f"✅ Instance is ACTIVE and {REQUIRED_INSTANCE_FIELD} is available.")
                break
        elapsed = int(time.time() - start_time)
        print(f"   ⏱️ Waiting for instance to be ACTIVE with {REQUIRED_INSTANCE_FIELD}... ({elapsed // 60}m {elapsed % 60:02d}s)", end="\r")
        time.sleep(20)

if NETWORKING_MODE == "PSC":
    if not TARGET_SERVICE:
        raise RuntimeError("❌ Failed to retrieve Service Attachment from Apigee Instance!")
    print(f"🔗 Service Attachment: {TARGET_SERVICE}")
    os.environ["TARGET_SERVICE"] = TARGET_SERVICE
elif NETWORKING_MODE == "VPC_PEERING":
    if not INSTANCE_HOST:
        raise RuntimeError("❌ Failed to retrieve the instance host (internal runtime IP) from Apigee Instance!")
    print(f"🏠 Instance host (internal runtime IP): {INSTANCE_HOST}")
    os.environ["INSTANCE_HOST"] = INSTANCE_HOST
    # serviceAttachment is optional in VPC_PEERING mode: do not raise if absent.
    os.environ["TARGET_SERVICE"] = TARGET_SERVICE or ""
    if TARGET_SERVICE:
        print(f"ℹ️ serviceAttachment also present (not used in VPC_PEERING mode): {TARGET_SERVICE}")
else:
    raise unsupported_mode_error()

## Step 5: Provision the Northbound HTTPS Load Balancer (PSC NEG or VPC-Peering MIG Bridge)

This step takes one of two mutually exclusive paths, depending on `NETWORKING_MODE`. Both paths reserve a global IP and end with the same **URL Map → Google-managed SSL Certificate → Target HTTPS Proxy → Global Forwarding Rule (443)** chain. Both also set the same `IP_ADDRESS` and `RUNTIME_HOST_ALIAS` (`<ip-with-dashes>.nip.io`) variables, so Step 6 runs unchanged in either mode.

#### 🔒 `PSC` path: Private Service Connect NEG (no VMs)
Instead of deploying heavy, costly Envoy MIG VMs, we configure a **Private Service Connect (PSC) Network Endpoint Group (NEG)**:
1. **Global External IPv4 Address**
2. **PSC NEG** pointing to the Apigee `serviceAttachment`
3. **Global Backend Service** (Protocol: HTTPS, Load Balancing Scheme: EXTERNAL_MANAGED)
4. **URL Map**
5. **Google-managed SSL Certificate** for `<ip-with-dashes>.nip.io`
6. **Target HTTPS Proxy** & **Global Forwarding Rule** (Port 443)

#### 🔗 `VPC_PEERING` path: Envoy MIG bridge
A global external load balancer can't use a peered Apigee instance as a backend directly. Traffic is bridged through a regional Managed Instance Group of Envoy VMs, configured by Apigee's startup script, which forwards to the instance `host`:
1. **Global External IPv4 Address**
2. **Firewall rule** allowing `tcp:443` from the Google LB & health-check ranges `130.211.0.0/22` and `35.191.0.0/16` to the bridge VMs (tag `gke-apigee-proxy`)
3. **Instance template + regional MIG** in `$REGION`: e2-medium, Debian 12, no external IP, `ENDPOINT=<instance host>`. Includes autoscaling (max 3) and named port `https:443`
4. **HTTPS health check** on `/healthz/ingress`
5. **Global Backend Service** (HTTPS, EXTERNAL_MANAGED, 302s timeout) with the MIG as its backend
6. **URL Map**, **Google-managed SSL Certificate**, **Target HTTPS Proxy** & **Global Forwarding Rule** (Port 443)

> [!NOTE]
> The bridge VMs belong to you: patching, scaling and monitoring are your responsibility. Apigee only provides the startup script.

In [ ]:
#@title 🛡️ Create External HTTPS LB (PSC NEG or VPC-Peering Envoy MIG Bridge)
if NETWORKING_MODE == "PSC":
    # =======================================================================
    # PSC path: PSC NEG -> Apigee serviceAttachment (unchanged)
    # =======================================================================
    # 1. Reserve Global IPv4 Address
    print("1. Reserving global external IPv4 address...")
    !gcloud compute addresses create apigee-ipaddress --ip-version=IPV4 --global --project=$PROJECT_ID --quiet || true

    IP_ADDRESS = subprocess.check_output(
        f"gcloud compute addresses describe apigee-ipaddress --format='value(address)' --global --project={PROJECT_ID}",
        shell=True, text=True
    ).strip()

    RUNTIME_HOST_ALIAS = f"{IP_ADDRESS.replace('.', '-')}.nip.io"
    os.environ["IP_ADDRESS"] = IP_ADDRESS
    os.environ["RUNTIME_HOST_ALIAS"] = RUNTIME_HOST_ALIAS

    print(f"📍 Reserved IP Address: {IP_ADDRESS}")
    print(f"🌐 Wildcard Hostname: {RUNTIME_HOST_ALIAS}")

    # 2. Create PSC NEG
    print(f"\n2. Creating Private Service Connect NEG in {REGION}...")
    !gcloud compute network-endpoint-groups create apigee-neg \
        --network-endpoint-type=private-service-connect \
        --psc-target-service=$TARGET_SERVICE \
        --region=$REGION \
        --network=$NETWORK_NAME \
        --project=$PROJECT_ID --quiet || true

    # 3. Create Backend Service
    print("\n3. Creating Global Backend Service...")
    !gcloud compute backend-services create apigee-backend \
        --load-balancing-scheme=EXTERNAL_MANAGED \
        --protocol=HTTPS \
        --global --project=$PROJECT_ID --quiet || true

    # Add PSC NEG to Backend Service
    print("\n4. Attaching PSC NEG to Backend Service...")
    !gcloud compute backend-services add-backend apigee-backend \
        --network-endpoint-group=apigee-neg \
        --network-endpoint-group-region=$REGION \
        --global --project=$PROJECT_ID --quiet || true

    # 5. Create URL Map
    print("\n5. Creating URL Map...")
    !gcloud compute url-maps create apigee-lb \
        --default-service=apigee-backend \
        --global --project=$PROJECT_ID --quiet || true

    # 6. Create Google-managed SSL Certificate
    print(f"\n6. Creating Google-managed SSL Certificate for {RUNTIME_HOST_ALIAS}...")
    !gcloud compute ssl-certificates create apigee-ssl-cert \
        --domains="$RUNTIME_HOST_ALIAS" \
        --project=$PROJECT_ID --quiet || true

    # 7. Create Target HTTPS Proxy
    print("\n7. Creating Target HTTPS Proxy...")
    !gcloud compute target-https-proxies create apigee-proxy \
        --url-map=apigee-lb \
        --ssl-certificates=apigee-ssl-cert \
        --project=$PROJECT_ID --quiet || true

    # 8. Create Global Forwarding Rule
    print("\n8. Creating Global Forwarding Rule on Port 443...")
    !gcloud compute forwarding-rules create apigee-fw-rule \
        --load-balancing-scheme=EXTERNAL_MANAGED \
        --network-tier=PREMIUM \
        --address=$IP_ADDRESS \
        --target-https-proxy=apigee-proxy \
        --ports=443 \
        --global --project=$PROJECT_ID --quiet || true

    print("\n✅ PSC External HTTPS Load Balancer provisioned successfully!")

elif NETWORKING_MODE == "VPC_PEERING":
    # =======================================================================
    # VPC_PEERING path: Envoy MIG bridge -> Apigee instance host (peered IP)
    # =======================================================================
    if not globals().get("INSTANCE_HOST"):
        raise RuntimeError("❌ INSTANCE_HOST is not set. Run Step 4 first.")

    MIG_NAME = f"apigee-mig-{REGION}"
    MIG_TEMPLATE = f"apigee-mig-template-{REGION}"
    MIG_HEALTH_CHECK = "apigee-mig-hc"
    MIG_BACKEND = "apigee-mig-backend"
    MIG_FIREWALL_RULE = "apigee-mig-allow-lb-hc"
    MIG_NETWORK_TAG = "gke-apigee-proxy"
    ENVOY_STARTUP_SCRIPT = "gs://apigee-5g-saas/apigee-envoy-proxy-release/latest/conf/startup-script.sh"
    os.environ["MIG_NAME"] = MIG_NAME
    os.environ["MIG_TEMPLATE"] = MIG_TEMPLATE
    os.environ["MIG_HEALTH_CHECK"] = MIG_HEALTH_CHECK
    os.environ["MIG_BACKEND"] = MIG_BACKEND
    os.environ["MIG_FIREWALL_RULE"] = MIG_FIREWALL_RULE
    os.environ["MIG_NETWORK_TAG"] = MIG_NETWORK_TAG
    os.environ["ENVOY_STARTUP_SCRIPT"] = ENVOY_STARTUP_SCRIPT

    # 1. Reserve Global IPv4 Address
    print("1. Reserving global external IPv4 address...")
    !gcloud compute addresses create apigee-ipaddress --ip-version=IPV4 --global --project=$PROJECT_ID --quiet || true

    IP_ADDRESS = subprocess.check_output(
        f"gcloud compute addresses describe apigee-ipaddress --format='value(address)' --global --project={PROJECT_ID}",
        shell=True, text=True
    ).strip()

    RUNTIME_HOST_ALIAS = f"{IP_ADDRESS.replace('.', '-')}.nip.io"
    os.environ["IP_ADDRESS"] = IP_ADDRESS
    os.environ["RUNTIME_HOST_ALIAS"] = RUNTIME_HOST_ALIAS

    print(f"📍 Reserved IP Address: {IP_ADDRESS}")
    print(f"🌐 Wildcard Hostname: {RUNTIME_HOST_ALIAS}")

    # Resolve the bridge subnet in $REGION (validated and PGA-enabled in Step 2)
    if not globals().get("MIG_SUBNET"):
        _subnets = json.loads(run_gcloud(
            f"gcloud compute networks subnets list --network={NETWORK_NAME} --project={PROJECT_ID} --format=json"
        ).stdout or "[]")
        _region_subnets = [s["name"] for s in _subnets if (s.get("region") or "").split("/")[-1] == REGION]
        if not _region_subnets:
            raise RuntimeError(f"❌ Network '{NETWORK_NAME}' has no subnet in {REGION}. Run Step 2 first.")
        MIG_SUBNET = _region_subnets[0]
    os.environ["MIG_SUBNET"] = MIG_SUBNET

    # 2. Firewall rule: Google LB proxies + health checks -> bridge VMs
    print(f"\n2. Creating firewall rule '{MIG_FIREWALL_RULE}' (130.211.0.0/22, 35.191.0.0/16 -> tcp:443)...")
    !gcloud compute firewall-rules create $MIG_FIREWALL_RULE \
        --description="Allow Google LB and health checks on TCP 443 to the Apigee Envoy MIG bridge" \
        --network=$NETWORK_NAME \
        --direction=INGRESS \
        --allow=tcp:443 \
        --source-ranges=130.211.0.0/22,35.191.0.0/16 \
        --target-tags=$MIG_NETWORK_TAG \
        --project=$PROJECT_ID --quiet || true

    # 3. Instance template + regional MIG running the Apigee Envoy bridge
    print(f"\n3. Creating Envoy bridge instance template '{MIG_TEMPLATE}' (ENDPOINT={INSTANCE_HOST}, subnet={MIG_SUBNET})...")
    tmpl_check = run_gcloud(
        f"gcloud compute instance-templates describe {MIG_TEMPLATE} --project={PROJECT_ID} --format=json",
        check=False,
    )
    if tmpl_check.returncode == 0:
        tmpl_items = json.loads(tmpl_check.stdout).get("properties", {}).get("metadata", {}).get("items", []) or []
        tmpl_endpoint = next((i.get("value") for i in tmpl_items if i.get("key") == "ENDPOINT"), None)
        if tmpl_endpoint != INSTANCE_HOST:
            print(f"⚠️ WARNING: Existing template targets ENDPOINT={tmpl_endpoint}, but the instance host is {INSTANCE_HOST}.")
            print("   Instance templates are immutable: delete the MIG and the template, then re-run this cell.")
        else:
            print(f"ℹ️ Instance template '{MIG_TEMPLATE}' already exists (reusing).")
    else:
        !gcloud compute instance-templates create $MIG_TEMPLATE \
            --project=$PROJECT_ID \
            --region=$REGION \
            --network=$NETWORK_NAME \
            --subnet=$MIG_SUBNET \
            --tags=https-server,apigee-mig-proxy,$MIG_NETWORK_TAG \
            --machine-type=e2-medium \
            --image-family=debian-12 \
            --image-project=debian-cloud \
            --boot-disk-size=20GB \
            --no-address \
            --metadata=ENDPOINT=$INSTANCE_HOST,startup-script-url=$ENVOY_STARTUP_SCRIPT \
            --quiet || true

    print(f"\n   Creating regional MIG '{MIG_NAME}' in {REGION}...")
    !gcloud compute instance-groups managed create $MIG_NAME \
        --project=$PROJECT_ID \
        --base-instance-name=apigee-mig \
        --size=2 \
        --template=$MIG_TEMPLATE \
        --region=$REGION --quiet || true

    !gcloud compute instance-groups managed set-autoscaling $MIG_NAME \
        --project=$PROJECT_ID --region=$REGION \
        --max-num-replicas=3 \
        --target-cpu-utilization=0.75 \
        --cool-down-period=90 --quiet || true

    !gcloud compute instance-groups managed set-named-ports $MIG_NAME \
        --project=$PROJECT_ID --region=$REGION \
        --named-ports=https:443 --quiet || true

    # 4. HTTPS health check against the Envoy bridge
    print(f"\n4. Creating HTTPS health check '{MIG_HEALTH_CHECK}' (/healthz/ingress)...")
    !gcloud compute health-checks create https $MIG_HEALTH_CHECK \
        --port=443 \
        --request-path=/healthz/ingress \
        --global --project=$PROJECT_ID --quiet || true

    # 5. Global Backend Service with the MIG as backend
    print(f"\n5. Creating Global Backend Service '{MIG_BACKEND}'...")
    !gcloud compute backend-services create $MIG_BACKEND \
        --load-balancing-scheme=EXTERNAL_MANAGED \
        --protocol=HTTPS \
        --health-checks=$MIG_HEALTH_CHECK \
        --port-name=https \
        --timeout=302s \
        --connection-draining-timeout=300s \
        --global --project=$PROJECT_ID --quiet || true

    print("   Attaching Envoy MIG to Backend Service...")
    !gcloud compute backend-services add-backend $MIG_BACKEND \
        --instance-group=$MIG_NAME \
        --instance-group-region=$REGION \
        --balancing-mode=UTILIZATION \
        --max-utilization=0.8 \
        --global --project=$PROJECT_ID --quiet || true

    # 6. Create URL Map
    print("\n6. Creating URL Map...")
    !gcloud compute url-maps create apigee-lb \
        --default-service=$MIG_BACKEND \
        --global --project=$PROJECT_ID --quiet || true

    # 7. Create Google-managed SSL Certificate
    print(f"\n7. Creating Google-managed SSL Certificate for {RUNTIME_HOST_ALIAS}...")
    !gcloud compute ssl-certificates create apigee-ssl-cert \
        --domains="$RUNTIME_HOST_ALIAS" \
        --project=$PROJECT_ID --quiet || true

    # 8. Create Target HTTPS Proxy
    print("\n8. Creating Target HTTPS Proxy...")
    !gcloud compute target-https-proxies create apigee-proxy \
        --url-map=apigee-lb \
        --ssl-certificates=apigee-ssl-cert \
        --project=$PROJECT_ID --quiet || true

    # 9. Create Global Forwarding Rule
    print("\n9. Creating Global Forwarding Rule on Port 443...")
    !gcloud compute forwarding-rules create apigee-fw-rule \
        --load-balancing-scheme=EXTERNAL_MANAGED \
        --network-tier=PREMIUM \
        --address=$IP_ADDRESS \
        --target-https-proxy=apigee-proxy \
        --ports=443 \
        --global --project=$PROJECT_ID --quiet || true

    # Wait (gcloud polls) until the bridge VMs are created and stable
    print(f"\n⏳ Waiting for MIG '{MIG_NAME}' to become stable...")
    !gcloud compute instance-groups managed wait-until $MIG_NAME --stable --region=$REGION --project=$PROJECT_ID --timeout=900

    print("\n✅ VPC-peering External HTTPS Load Balancer (Envoy MIG bridge) provisioned successfully!")

else:
    raise unsupported_mode_error()

# Both paths set the same variables consumed by Step 6
if not (globals().get("IP_ADDRESS") and globals().get("RUNTIME_HOST_ALIAS")):
    raise RuntimeError("❌ IP_ADDRESS / RUNTIME_HOST_ALIAS were not set by the load balancer step.")
print(f"\n📌 IP_ADDRESS={IP_ADDRESS} | RUNTIME_HOST_ALIAS={RUNTIME_HOST_ALIAS}")

## Step 6: Create Environment, Environment Group & Attachments

In this step, we configure Apigee routing:
1. **Environment Group**: Binds the hostname (`<ip>.nip.io`) to environments.
2. **Environment**: Execution context (`dev`) where API proxies are deployed.
3. **Envgroup Attachment**: Connects the `dev` environment to `dev-group`.
4. **Instance Attachment**: Connects the `dev` environment to the instance so the runtime can execute proxies.
5. **Trace Config**: Enables Cloud Trace sampling for distributed tracing.


In [ ]:
#@title 🔗 Configure Environment, Environment Group & Attachments
# 1. Create Environment Group
print(f"1. Creating Environment Group '{ENVGROUP_NAME}' with hostname '{RUNTIME_HOST_ALIAS}'...")
envgroup_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/envgroups"
eg_resp = requests.post(
    envgroup_url,
    headers=apigee_headers(),
    json={"name": ENVGROUP_NAME, "hostnames": [RUNTIME_HOST_ALIAS]}
)
if eg_resp.status_code in (200, 201):
    op = eg_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Envgroup Creation")
elif "already exists" in eg_resp.text:
    print(f"Envgroup '{ENVGROUP_NAME}' already exists.")
else:
    print(f"Envgroup response: {eg_resp.status_code} - {eg_resp.text}")

# 2. Create Environment
print(f"\n2. Creating Environment '{ENV_NAME}'...")
env_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/environments"
env_resp = requests.post(
    env_url,
    headers=apigee_headers(),
    json={"name": ENV_NAME}
)
if env_resp.status_code in (200, 201):
    op = env_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Environment Creation")
elif "already exists" in env_resp.text:
    print(f"Environment '{ENV_NAME}' already exists.")
else:
    print(f"Environment response: {env_resp.status_code} - {env_resp.text}")

# 3. Attach Environment to Envgroup
print(f"\n3. Attaching Environment '{ENV_NAME}' to Envgroup '{ENVGROUP_NAME}'...")
eg_att_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/envgroups/{ENVGROUP_NAME}/attachments"
eg_att_resp = requests.post(
    eg_att_url,
    headers=apigee_headers(),
    json={"name": f"{ENVGROUP_NAME}-{ENV_NAME}", "environment": ENV_NAME}
)
if eg_att_resp.status_code in (200, 201):
    op = eg_att_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Envgroup Attachment")
elif "already exists" in eg_att_resp.text:
    print("Environment is already attached to Envgroup.")

# 4. Attach Environment to Instance
print(f"\n4. Attaching Environment '{ENV_NAME}' to Instance '{INSTANCE_NAME}'...")
inst_att_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances/{INSTANCE_NAME}/attachments"
inst_att_resp = requests.post(
    inst_att_url,
    headers=apigee_headers(),
    json={"environment": ENV_NAME}
)
if inst_att_resp.status_code in (200, 201):
    op = inst_att_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Instance Attachment")
elif "already exists" in inst_att_resp.text:
    print("Environment is already attached to Instance.")

# 5. Configure Cloud Trace
print("\n5. Configuring Cloud Trace exporter...")
trace_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/environments/{ENV_NAME}/traceConfig"
requests.patch(
    trace_url,
    headers=apigee_headers(),
    json={
        "exporter": "CLOUD_TRACE",
        "endpoint": PROJECT_ID,
        "samplingConfig": {"sampler": "PROBABILITY", "samplingRate": 0.5}
    }
)

print("✅ Environment routing and attachments configured successfully!")


## Step 7: Summary & Verification

Congratulations! Your Apigee organization, instance, and load balancer are fully provisioned and ready for proxy deployment.

The summary shows the **Networking Mode**. In `VPC_PEERING` mode it also shows the allocated peering and support CIDRs, the instance host IP, and the Envoy MIG bridge.

> [!NOTE]
> Google-managed SSL certificates take around **10 to 20 minutes** after provisioning to become `ACTIVE` once Google verifies domain routing.

In [ ]:
#@title 🎉 Deployment Summary & Test
ssl_status = subprocess.check_output(
    f"gcloud compute ssl-certificates describe apigee-ssl-cert --format='value(managed.status)' --project={PROJECT_ID}",
    shell=True, text=True
).strip()

print("=" * 60)
print("🎉 APIGEE PROVISIONING COMPLETE!")
print("=" * 60)
print(f"📌 Project ID:         {PROJECT_ID}")
print(f"🏛️ Apigee Org:         {PROJECT_ID}")
print(f"💳 Billing Type:       {BILLING_TYPE}")
print(f"🔀 Networking Mode:    {NETWORKING_MODE}")
print(f"📍 Instance:           {INSTANCE_NAME} ({REGION})")
print(f"🌿 Environment:        {ENV_NAME}")
print(f"🌐 Environment Group:  {ENVGROUP_NAME}")
print(f"🔗 Public IP:          {IP_ADDRESS}")
print(f"🌍 Hostname (nip.io):  https://{RUNTIME_HOST_ALIAS}")
print(f"🔒 SSL Cert Status:    {ssl_status} (will transition to ACTIVE in 10-20 min)")
if NETWORKING_MODE == "PSC":
    print(f"🔌 PSC Attachment:     {globals().get('TARGET_SERVICE') or 'n/a'}")
    print("📐 Peering Ranges:     none consumed (disableVpcPeering=true)")
elif NETWORKING_MODE == "VPC_PEERING":
    print(f"📐 Peering CIDR:       {globals().get('PEERING_CIDR') or 'n/a'} ({PEERING_RANGE_NAME}, {PEERING_CIDR_SIZE})")
    print(f"🧰 Support CIDR:       {globals().get('SUPPORT_CIDR') or 'n/a'} ({SUPPORT_RANGE_NAME}, /28)")
    print(f"🏠 Instance Host IP:   {globals().get('INSTANCE_HOST') or 'n/a'}")
    print(f"🖥️ Envoy MIG Bridge:   {globals().get('MIG_NAME') or 'n/a'} ({REGION})")
else:
    raise unsupported_mode_error()
print("=" * 60)
print("\n👉 Apigee Console URL:")
print(f"https://console.cloud.google.com/apigee/proxies?project={PROJECT_ID}")